In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/antonkozyriev/game-recommendations-on-steam/games_metadata.json
/kaggle/input/datasets/antonkozyriev/game-recommendations-on-steam/users.csv
/kaggle/input/datasets/antonkozyriev/game-recommendations-on-steam/games.csv
/kaggle/input/datasets/antonkozyriev/game-recommendations-on-steam/recommendations.csv


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.10.0+cu128
True
2


In [3]:
!pip install torch_geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 886.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.6 MB/s eta 0:00:00


In [4]:
import os

path = "/kaggle/input/datasets/antonkozyriev/game-recommendations-on-steam"
print("Inside game-recommendations-on-steam:")
print(os.listdir(path))

Inside game-recommendations-on-steam:
['games_metadata.json', 'users.csv', 'games.csv', 'recommendations.csv']


In [5]:
import pandas as pd

data_path = "/kaggle/input/datasets/antonkozyriev/game-recommendations-on-steam"

games = pd.read_csv(f"{data_path}/games.csv")
recommendations = pd.read_csv(f"{data_path}/recommendations.csv")

print("games.csv shape:", games.shape)
print("recommendations.csv shape:", recommendations.shape)

games.csv shape: (50872, 13)
recommendations.csv shape: (41154794, 8)


In [6]:
user_counts = recommendations["user_id"].value_counts()
recommendations["user_review_count"] = recommendations["user_id"].map(user_counts)

THRESHOLD = 50
filtered = recommendations[recommendations["user_review_count"] >= THRESHOLD].copy()

unique_users = filtered["user_id"].unique()
unique_games = filtered["app_id"].unique()

user_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_users)}
game_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_games)}

filtered["user_node_id"] = filtered["user_id"].map(user_id_map)
filtered["game_node_id"] = filtered["app_id"].map(game_id_map)

output = filtered[["user_node_id", "game_node_id", "is_recommended", "hours", "user_id", "app_id"]]
output.to_csv("/kaggle/working/filtered_edges_dev.csv", index=False)

print("Number of user nodes:", len(unique_users))
print("Number of game nodes:", len(unique_games))
print("Number of edges:", len(output))

Number of user nodes: 37160
Number of game nodes: 36229
Number of edges: 3439762


In [7]:
import torch
from torch_geometric.data import HeteroData

edges = pd.read_csv("/kaggle/working/filtered_edges_dev.csv")

n_users = edges["user_node_id"].nunique()
n_games = edges["game_node_id"].nunique()

data = HeteroData()
data['user'].num_nodes = n_users
data['game'].num_nodes = n_games

user_ids = torch.tensor(edges["user_node_id"].values, dtype=torch.long)
game_ids = torch.tensor(edges["game_node_id"].values, dtype=torch.long)

edge_index = torch.stack([user_ids, game_ids], dim=0)
data['user', 'reviews', 'game'].edge_index = edge_index

is_recommended = torch.tensor(edges["is_recommended"].astype(int).values, dtype=torch.float).unsqueeze(1)
hours = torch.tensor(edges["hours"].values, dtype=torch.float)
hours_norm = ((hours - hours.mean()) / (hours.std() + 1e-8)).unsqueeze(1)

edge_attr = torch.cat([is_recommended, hours_norm], dim=1)
data['user', 'reviews', 'game'].edge_attr = edge_attr

data['game', 'rev_reviews', 'user'].edge_index = torch.stack([game_ids, user_ids], dim=0)
data['game', 'rev_reviews', 'user'].edge_attr = edge_attr

print(data)

torch.save(data, "/kaggle/working/graph_dev.pt")
print("Saved graph object to graph_dev.pt")

HeteroData(
  user={ num_nodes=37160 },
  game={ num_nodes=36229 },
  (user, reviews, game)={
    edge_index=[2, 3439762],
    edge_attr=[3439762, 2],
  },
  (game, rev_reviews, user)={
    edge_index=[2, 3439762],
    edge_attr=[3439762, 2],
  }
)
Saved graph object to graph_dev.pt


In [8]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

EMBEDDING_DIM = 64

user_embedding = nn.Embedding(num_embeddings=n_users, embedding_dim=EMBEDDING_DIM).to(device)
game_embedding = nn.Embedding(num_embeddings=n_games, embedding_dim=EMBEDDING_DIM).to(device)

print("User embedding table shape:", user_embedding.weight.shape)
print("Game embedding table shape:", game_embedding.weight.shape)

Using device: cuda
User embedding table shape: torch.Size([37160, 64])
Game embedding table shape: torch.Size([36229, 64])


In [9]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, to_hetero

class GNNEncoder(nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

gnn_encoder = GNNEncoder(hidden_channels=64, out_channels=64)
gnn_encoder = to_hetero(gnn_encoder, data.metadata(), aggr='sum').to(device)

data = data.to(device)

x_dict = {
    'user': user_embedding.weight,
    'game': game_embedding.weight,
}
edge_index_dict = data.edge_index_dict

out_dict = gnn_encoder(x_dict, edge_index_dict)

print("Updated user embeddings shape:", out_dict['user'].shape)
print("Updated game embeddings shape:", out_dict['game'].shape)

Updated user embeddings shape: torch.Size([37160, 64])
Updated game embeddings shape: torch.Size([36229, 64])


In [10]:
from torch_geometric.utils import negative_sampling

def compute_loss(out_dict, edge_index):
    user_emb = out_dict['user']
    game_emb = out_dict['game']

    pos_user = edge_index[0]
    pos_game = edge_index[1]
    pos_score = (user_emb[pos_user] * game_emb[pos_game]).sum(dim=-1)

    neg_edge_index = negative_sampling(
        edge_index=edge_index,
        num_nodes=(user_emb.size(0), game_emb.size(0)),
        num_neg_samples=pos_user.size(0)
    )
    neg_user = neg_edge_index[0]
    neg_game = neg_edge_index[1]
    neg_score = (user_emb[neg_user] * game_emb[neg_game]).sum(dim=-1)

    pos_labels = torch.ones_like(pos_score)
    neg_labels = torch.zeros_like(neg_score)

    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([pos_labels, neg_labels])

    loss = F.binary_cross_entropy_with_logits(scores, labels)
    return loss

edge_index = data['user', 'reviews', 'game'].edge_index
loss = compute_loss(out_dict, edge_index)

print("Loss on one forward pass:", loss.item())

Loss on one forward pass: 0.735214114189148


In [11]:
import torch.optim as optim

params = list(gnn_encoder.parameters()) + list(user_embedding.parameters()) + list(game_embedding.parameters())
optimizer = optim.Adam(params, lr=0.01)

EPOCHS = 100

for epoch in range(1, EPOCHS + 1):
    gnn_encoder.train()
    optimizer.zero_grad()

    x_dict = {
        'user': user_embedding.weight,
        'game': game_embedding.weight,
    }
    edge_index_dict = data.edge_index_dict

    out_dict = gnn_encoder(x_dict, edge_index_dict)

    loss = compute_loss(out_dict, edge_index)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7354
Epoch 10, Loss: 0.6841
Epoch 20, Loss: 0.4181
Epoch 30, Loss: 0.3465
Epoch 40, Loss: 0.3282
Epoch 50, Loss: 0.3193
Epoch 60, Loss: 0.3136
Epoch 70, Loss: 0.3093
Epoch 80, Loss: 0.3037
Epoch 90, Loss: 0.2984
Epoch 100, Loss: 0.2927


In [12]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.05,
    is_undirected=False,
    edge_types=('user', 'reviews', 'game'),
    rev_edge_types=('game', 'rev_reviews', 'user'),
    neg_sampling_ratio=1.0,
)

train_data, val_data, test_data = transform(data)

print(train_data)
print(val_data)
print(test_data)

HeteroData(
  user={ num_nodes=37160 },
  game={ num_nodes=36229 },
  (user, reviews, game)={
    edge_index=[2, 2923798],
    edge_attr=[2923798, 2],
    edge_label=[5847596],
    edge_label_index=[2, 5847596],
  },
  (game, rev_reviews, user)={
    edge_index=[2, 2923798],
    edge_attr=[2923798, 2],
  }
)
HeteroData(
  user={ num_nodes=37160 },
  game={ num_nodes=36229 },
  (user, reviews, game)={
    edge_index=[2, 2923798],
    edge_attr=[2923798, 2],
    edge_label=[687952],
    edge_label_index=[2, 687952],
  },
  (game, rev_reviews, user)={
    edge_index=[2, 2923798],
    edge_attr=[2923798, 2],
  }
)
HeteroData(
  user={ num_nodes=37160 },
  game={ num_nodes=36229 },
  (user, reviews, game)={
    edge_index=[2, 3267774],
    edge_attr=[3267774, 2],
    edge_label=[343976],
    edge_label_index=[2, 343976],
  },
  (game, rev_reviews, user)={
    edge_index=[2, 3267774],
    edge_attr=[3267774, 2],
  }
)


In [13]:
from sklearn.metrics import roc_auc_score

EMBEDDING_DIM = 64

user_embedding = nn.Embedding(num_embeddings=n_users, embedding_dim=EMBEDDING_DIM).to(device)
game_embedding = nn.Embedding(num_embeddings=n_games, embedding_dim=EMBEDDING_DIM).to(device)

gnn_encoder = GNNEncoder(hidden_channels=64, out_channels=64)
gnn_encoder = to_hetero(gnn_encoder, data.metadata(), aggr='sum').to(device)

train_data = train_data.to(device)
val_data = val_data.to(device)

params = list(gnn_encoder.parameters()) + list(user_embedding.parameters()) + list(game_embedding.parameters())
optimizer = optim.Adam(params, lr=0.01)

def score_edges(out_dict, edge_label_index):
    user_idx = edge_label_index[0]
    game_idx = edge_label_index[1]
    return (out_dict['user'][user_idx] * out_dict['game'][game_idx]).sum(dim=-1)

EPOCHS = 100

for epoch in range(1, EPOCHS + 1):
    gnn_encoder.train()
    optimizer.zero_grad()

    x_dict = {
        'user': user_embedding.weight,
        'game': game_embedding.weight,
    }
    out_dict = gnn_encoder(x_dict, train_data.edge_index_dict)

    edge_label_index = train_data['user', 'reviews', 'game'].edge_label_index
    edge_label = train_data['user', 'reviews', 'game'].edge_label

    scores = score_edges(out_dict, edge_label_index)
    loss = F.binary_cross_entropy_with_logits(scores, edge_label)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}")

gnn_encoder.eval()
with torch.no_grad():
    x_dict = {
        'user': user_embedding.weight,
        'game': game_embedding.weight,
    }
    out_dict = gnn_encoder(x_dict, val_data.edge_index_dict)

    val_edge_label_index = val_data['user', 'reviews', 'game'].edge_label_index
    val_edge_label = val_data['user', 'reviews', 'game'].edge_label

    val_scores = score_edges(out_dict, val_edge_label_index).cpu().numpy()
    val_labels = val_edge_label.cpu().numpy()

    auc = roc_auc_score(val_labels, val_scores)
    print(f"Validation AUC: {auc:.4f}")

Epoch 1, Train Loss: 0.7350
Epoch 10, Train Loss: 0.7416
Epoch 20, Train Loss: 0.5458
Epoch 30, Train Loss: 0.4000
Epoch 40, Train Loss: 0.3431
Epoch 50, Train Loss: 0.3258
Epoch 60, Train Loss: 0.3191
Epoch 70, Train Loss: 0.3139
Epoch 80, Train Loss: 0.3093
Epoch 90, Train Loss: 0.3051
Epoch 100, Train Loss: 0.3008
Validation AUC: 0.9385


In [14]:
test_data = test_data.to(device)

gnn_encoder.eval()
with torch.no_grad():
    x_dict = {
        'user': user_embedding.weight,
        'game': game_embedding.weight,
    }
    out_dict = gnn_encoder(x_dict, test_data.edge_index_dict)

    test_edge_label_index = test_data['user', 'reviews', 'game'].edge_label_index
    test_edge_label = test_data['user', 'reviews', 'game'].edge_label

    test_scores = score_edges(out_dict, test_edge_label_index).cpu().numpy()
    test_labels = test_edge_label.cpu().numpy()

    test_auc = roc_auc_score(test_labels, test_scores)
    print(f"Test AUC: {test_auc:.4f}")

Test AUC: 0.9390


In [15]:
sample_user_node_id = 0

with torch.no_grad():
    x_dict = {
        'user': user_embedding.weight,
        'game': game_embedding.weight,
    }
    out_dict = gnn_encoder(x_dict, data.edge_index_dict)

    user_vec = out_dict['user'][sample_user_node_id]
    all_game_vecs = out_dict['game']

    scores = (all_game_vecs * user_vec).sum(dim=-1)
    top_k = torch.topk(scores, k=10)

    top_game_node_ids = top_k.indices.cpu().numpy()

reverse_game_map = {v: k for k, v in game_id_map.items()}
top_app_ids = [reverse_game_map[gid] for gid in top_game_node_ids]

top_games = games[games["app_id"].isin(top_app_ids)][["app_id", "title"]]
print(top_games)

        app_id                              title
480     552520                         Far Cry® 5
11718   242760                         The Forest
12711      550                      Left 4 Dead 2
12800   292030          The Witcher® 3: Wild Hunt
13181   377160                          Fallout 4
13598   976730  Halo: The Master Chief Collection
14163  1091500                     Cyberpunk 2077
15097   782330                       DOOM Eternal
41755   778080                           40 Winks
47730   379720                               DOOM


In [16]:
sample_user_ids = [0, 500, 10000]

reverse_game_map = {v: k for k, v in game_id_map.items()}

with torch.no_grad():
    x_dict = {
        'user': user_embedding.weight,
        'game': game_embedding.weight,
    }
    out_dict = gnn_encoder(x_dict, data.edge_index_dict)
    all_game_vecs = out_dict['game']

    recommendations_by_user = {}
    for uid in sample_user_ids:
        user_vec = out_dict['user'][uid]
        scores = (all_game_vecs * user_vec).sum(dim=-1)
        top_k = torch.topk(scores, k=10)
        top_game_node_ids = top_k.indices.cpu().numpy()
        top_app_ids = [reverse_game_map[gid] for gid in top_game_node_ids]
        recommendations_by_user[uid] = set(top_app_ids)
        titles = games[games["app_id"].isin(top_app_ids)]["title"].tolist()
        print(f"User {uid} top 10:", titles)
        print()

overlap_0_500 = recommendations_by_user[0] & recommendations_by_user[500]
overlap_0_10000 = recommendations_by_user[0] & recommendations_by_user[10000]
overlap_500_10000 = recommendations_by_user[500] & recommendations_by_user[10000]

print("Overlap between user 0 and user 500:", len(overlap_0_500), "out of 10")
print("Overlap between user 0 and user 10000:", len(overlap_0_10000), "out of 10")
print("Overlap between user 500 and user 10000:", len(overlap_500_10000), "out of 10")

User 0 top 10: ['Far Cry® 5', 'The Forest', 'Left 4 Dead 2', 'The Witcher® 3: Wild Hunt', 'Fallout 4', 'Halo: The Master Chief Collection', 'Cyberpunk 2077', 'DOOM Eternal', '40 Winks', 'DOOM']

User 500 top 10: ['The Forest', 'Left 4 Dead 2', 'Grand Theft Auto V', 'Fallout 4', 'Cyberpunk 2077', 'DOOM Eternal', '40 Winks', 'BioShock Infinite', 'Borderlands 2', 'DOOM']

User 10000 top 10: ['The Forest', 'Fallout 4', 'Halo: The Master Chief Collection', 'Cyberpunk 2077', 'DOOM Eternal', 'DARK SOULS™: REMASTERED', '40 Winks', 'Saints Row: The Third', 'Life is Strange - Episode 1', 'Rise of the Tomb Raider™']

Overlap between user 0 and user 500: 7 out of 10
Overlap between user 0 and user 10000: 6 out of 10
Overlap between user 500 and user 10000: 5 out of 10


In [17]:
import json

metadata_path = f"{data_path}/games_metadata.json"

metadata_list = []
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        metadata_list.append(json.loads(line))

metadata_df = pd.DataFrame(metadata_list)

print("Metadata shape:", metadata_df.shape)
print("Metadata columns:", metadata_df.columns.tolist())
print(metadata_df.head(3))

kept_app_ids = set(unique_games)
metadata_filtered = metadata_df[metadata_df["app_id"].isin(kept_app_ids)].copy()

print()
print("Games with metadata:", metadata_filtered.shape[0], "out of", len(kept_app_ids), "total filtered games")

Metadata shape: (50872, 3)
Metadata columns: ['app_id', 'description', 'tags']
   app_id                                        description  \
0   13500  Enter the dark underworld of Prince of Persia ...   
1   22364                                                      
2  113020  Monaco: What's Yours Is Mine is a single playe...   

                                                tags  
0  [Action, Adventure, Parkour, Third Person, Gre...  
1                                           [Action]  
2  [Co-op, Stealth, Indie, Heist, Local Co-Op, St...  

Games with metadata: 36229 out of 36229 total filtered games


In [18]:
from sentence_transformers import SentenceTransformer

nlp_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

sample_texts = [
    "A fast-paced roguelike with permadeath and procedurally generated dungeons",
    "An open world RPG with deep character customization and branching dialogue",
    "A fast-paced roguelike with permadeath and procedurally generated dungeons",
]

sample_embeddings = nlp_model.encode(sample_texts, convert_to_tensor=True)
print("Embedding shape:", sample_embeddings.shape)

from torch.nn.functional import cosine_similarity

sim_different = cosine_similarity(sample_embeddings[0].unsqueeze(0), sample_embeddings[1].unsqueeze(0))
sim_same = cosine_similarity(sample_embeddings[0].unsqueeze(0), sample_embeddings[2].unsqueeze(0))

print("Similarity between different genres:", sim_different.item())
print("Similarity between identical text:", sim_same.item())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: torch.Size([3, 384])
Similarity between different genres: 0.3645417094230652
Similarity between identical text: 1.0


In [19]:
reverse_game_map = {v: k for k, v in game_id_map.items()}

metadata_lookup = metadata_filtered.set_index("app_id")
titles_lookup = games.set_index("app_id")["title"]

ordered_texts = []
fallback_count = {"description": 0, "tags": 0, "title": 0}

for node_id in range(n_games):
    app_id = reverse_game_map[node_id]
    row = metadata_lookup.loc[app_id]

    description = row["description"]
    tags = row["tags"]

    if isinstance(description, str) and description.strip() != "":
        text = description
        fallback_count["description"] += 1
    elif isinstance(tags, list) and len(tags) > 0:
        text = ", ".join(tags)
        fallback_count["tags"] += 1
    else:
        text = titles_lookup.loc[app_id]
        fallback_count["title"] += 1

    ordered_texts.append(text)

print("Text source breakdown:")
print(fallback_count)
print()
print("Sample text, node 0:", ordered_texts[0][:200])

Text source breakdown:
{'description': 35619, 'tags': 35, 'title': 575}

Sample text, node 0: Barotrauma


In [20]:
game_text_embeddings = nlp_model.encode(
    ordered_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    device=device
)

print("NLP embedding table shape:", game_text_embeddings.shape)

torch.save(game_text_embeddings, "/kaggle/working/game_text_embeddings.pt")
print("Saved to game_text_embeddings.pt")

Batches:   0%|          | 0/284 [00:00<?, ?it/s]

NLP embedding table shape: torch.Size([36229, 384])
Saved to game_text_embeddings.pt


In [21]:
def get_game_node_id(title):
    matches = games[games["title"] == title]
    app_id = matches.iloc[0]["app_id"]
    return game_id_map[app_id]

fallout4_id = get_game_node_id("Fallout 4")
newvegas_id = get_game_node_id("Fallout: New Vegas")
portal2_id = get_game_node_id("Portal 2")

fallout4_vec = game_text_embeddings[fallout4_id].unsqueeze(0)
newvegas_vec = game_text_embeddings[newvegas_id].unsqueeze(0)
portal2_vec = game_text_embeddings[portal2_id].unsqueeze(0)

sim_same_franchise = cosine_similarity(fallout4_vec, newvegas_vec)
sim_different_genre = cosine_similarity(fallout4_vec, portal2_vec)

print("Fallout 4 vs Fallout New Vegas similarity:", sim_same_franchise.item())
print("Fallout 4 vs Portal 2 similarity:", sim_different_genre.item())

Fallout 4 vs Fallout New Vegas similarity: 0.8168131113052368
Fallout 4 vs Portal 2 similarity: 0.4204096794128418


In [22]:
import requests
from PIL import Image
from io import BytesIO

def get_header_image(app_id):
    url = f"https://cdn.akamai.steamstatic.com/steam/apps/{app_id}/header.jpg"
    response = requests.get(url, timeout=10)
    if response.status_code == 200:
        return Image.open(BytesIO(response.content)).convert("RGB")
    else:
        return None

test_app_ids = [377160, 22380, 1091500, 620, 4000]

for app_id in test_app_ids:
    img = get_header_image(app_id)
    if img is not None:
        print(f"app_id {app_id}: loaded, size {img.size}")
    else:
        print(f"app_id {app_id}: FAILED to load")

app_id 377160: loaded, size (460, 215)
app_id 22380: loaded, size (460, 215)
app_id 1091500: loaded, size (460, 215)
app_id 620: loaded, size (460, 215)
app_id 4000: loaded, size (460, 215)


In [23]:
import torchvision.models as tv_models
from concurrent.futures import ThreadPoolExecutor
import numpy as np

cv_model = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
cv_model.fc = nn.Identity()
cv_model = cv_model.to(device)
cv_model.eval()

preprocess = tv_models.ResNet18_Weights.DEFAULT.transforms()

def download_one(app_id):
    try:
        url = f"https://cdn.akamai.steamstatic.com/steam/apps/{app_id}/header.jpg"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return app_id, Image.open(BytesIO(response.content)).convert("RGB")
        else:
            return app_id, None
    except Exception:
        return app_id, None

def encode_batch(app_ids, batch_size=64, max_workers=32):
    all_embeddings = np.zeros((len(app_ids), 512), dtype=np.float32)
    failed_app_ids = []

    for start in range(0, len(app_ids), batch_size):
        batch_ids = app_ids[start:start + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            results = list(executor.map(download_one, batch_ids))

        valid_images = []
        valid_positions = []
        for i, (app_id, img) in enumerate(results):
            if img is not None:
                valid_images.append(preprocess(img))
                valid_positions.append(start + i)
            else:
                failed_app_ids.append(app_id)

        if valid_images:
            batch_tensor = torch.stack(valid_images).to(device)
            with torch.no_grad():
                batch_embeddings = cv_model(batch_tensor).cpu().numpy()
            for pos, emb in zip(valid_positions, batch_embeddings):
                all_embeddings[pos] = emb

    return all_embeddings, failed_app_ids

sample_app_ids = [reverse_game_map[nid] for nid in range(200)]

import time
start_time = time.time()
sample_embeddings, sample_failed = encode_batch(sample_app_ids)
elapsed = time.time() - start_time

print("Sample embeddings shape:", sample_embeddings.shape)
print("Failed downloads:", len(sample_failed), "out of 200")
print("Time taken:", round(elapsed, 1), "seconds")
print("Estimated time for all 36229 games:", round(elapsed / 200 * 36229 / 60, 1), "minutes")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 228MB/s]


Sample embeddings shape: (200, 512)
Failed downloads: 0 out of 200
Time taken: 2.1 seconds
Estimated time for all 36229 games: 6.2 minutes


In [24]:
all_app_ids_ordered = [reverse_game_map[nid] for nid in range(n_games)]

start_time = time.time()
game_image_embeddings, failed_app_ids = encode_batch(all_app_ids_ordered, batch_size=64, max_workers=32)
elapsed = time.time() - start_time

print("Full CV embedding table shape:", game_image_embeddings.shape)
print("Total failed downloads:", len(failed_app_ids), "out of", n_games)
print("Time taken:", round(elapsed / 60, 1), "minutes")

game_image_embeddings_tensor = torch.tensor(game_image_embeddings, dtype=torch.float32)
torch.save(game_image_embeddings_tensor, "/kaggle/working/game_image_embeddings.pt")
print("Saved to game_image_embeddings.pt")

Full CV embedding table shape: (36229, 512)
Total failed downloads: 3 out of 36229
Time taken: 5.2 minutes
Saved to game_image_embeddings.pt


In [25]:
def get_image_vec(app_id):
    node_id = game_id_map[app_id]
    return game_image_embeddings_tensor[node_id].unsqueeze(0)

darksouls_id = get_game_node_id("DARK SOULS™ III")
fallout4_id_img = get_game_node_id("Fallout 4")
tf2_id = get_game_node_id("Team Fortress 2")

darksouls_vec = get_image_vec(games[games["title"] == "DARK SOULS™ III"].iloc[0]["app_id"])
fallout4_vec_img = get_image_vec(games[games["title"] == "Fallout 4"].iloc[0]["app_id"])
tf2_vec = get_image_vec(games[games["title"] == "Team Fortress 2"].iloc[0]["app_id"])

sim_dark_games = cosine_similarity(darksouls_vec, fallout4_vec_img)
sim_dark_vs_bright = cosine_similarity(darksouls_vec, tf2_vec)

print("Dark Souls III vs Fallout 4 (both dark, gritty games):", sim_dark_games.item())
print("Dark Souls III vs Team Fortress 2 (dark vs bright, cartoonish):", sim_dark_games.item())
print("Dark Souls III vs Team Fortress 2 similarity:", sim_dark_vs_bright.item())

Dark Souls III vs Fallout 4 (both dark, gritty games): 0.5624375343322754
Dark Souls III vs Team Fortress 2 (dark vs bright, cartoonish): 0.5624375343322754
Dark Souls III vs Team Fortress 2 similarity: 0.6644423007965088


In [26]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

def encode_batch_clip(app_ids, batch_size=64, max_workers=32):
    all_embeddings = np.zeros((len(app_ids), 512), dtype=np.float32)
    failed_app_ids = []

    for start in range(0, len(app_ids), batch_size):
        batch_ids = app_ids[start:start + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            results = list(executor.map(download_one, batch_ids))

        valid_images = []
        valid_positions = []
        for i, (app_id, img) in enumerate(results):
            if img is not None:
                valid_images.append(img)
                valid_positions.append(start + i)
            else:
                failed_app_ids.append(app_id)

        if valid_images:
            inputs = clip_processor(images=valid_images, return_tensors="pt").to(device)
            with torch.no_grad():
                vision_outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
                pooled_output = vision_outputs.pooler_output
                batch_embeddings = clip_model.visual_projection(pooled_output).cpu().numpy()
            for pos, emb in zip(valid_positions, batch_embeddings):
                all_embeddings[pos] = emb

    return all_embeddings, failed_app_ids

sample_app_ids = [reverse_game_map[nid] for nid in range(200)]

start_time = time.time()
sample_clip_embeddings, sample_failed_clip = encode_batch_clip(sample_app_ids)
elapsed = time.time() - start_time

print("Sample CLIP embeddings shape:", sample_clip_embeddings.shape)
print("Failed downloads:", len(sample_failed_clip))
print("Time taken:", round(elapsed, 1), "seconds")
print("Estimated time for all 36229 games:", round(elapsed / 200 * 36229 / 60, 1), "minutes")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

... (3 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Sample CLIP embeddings shape: (200, 512)
Failed downloads: 0
Time taken: 2.2 seconds
Estimated time for all 36229 games: 6.5 minutes


In [27]:
start_time = time.time()
game_clip_embeddings, failed_clip_app_ids = encode_batch_clip(all_app_ids_ordered, batch_size=64, max_workers=32)
elapsed = time.time() - start_time

print("Full CLIP embedding table shape:", game_clip_embeddings.shape)
print("Total failed downloads:", len(failed_clip_app_ids), "out of", n_games)
print("Time taken:", round(elapsed / 60, 1), "minutes")

game_clip_embeddings_tensor = torch.tensor(game_clip_embeddings, dtype=torch.float32)
torch.save(game_clip_embeddings_tensor, "/kaggle/working/game_clip_embeddings.pt")
print("Saved to game_clip_embeddings.pt")

Full CLIP embedding table shape: (36229, 512)
Total failed downloads: 1 out of 36229
Time taken: 5.7 minutes
Saved to game_clip_embeddings.pt


In [28]:
def get_clip_image_vec(app_id):
    node_id = game_id_map[app_id]
    return game_clip_embeddings_tensor[node_id].unsqueeze(0)

darksouls_app_id = games[games["title"] == "DARK SOULS™ III"].iloc[0]["app_id"]
fallout4_app_id = games[games["title"] == "Fallout 4"].iloc[0]["app_id"]
tf2_app_id = games[games["title"] == "Team Fortress 2"].iloc[0]["app_id"]

darksouls_clip_vec = get_clip_image_vec(darksouls_app_id)
fallout4_clip_vec = get_clip_image_vec(fallout4_app_id)
tf2_clip_vec = get_clip_image_vec(tf2_app_id)

sim_dark_games_clip = cosine_similarity(darksouls_clip_vec, fallout4_clip_vec)
sim_dark_vs_bright_clip = cosine_similarity(darksouls_clip_vec, tf2_clip_vec)

print("CLIP: Dark Souls III vs Fallout 4 (both dark, gritty):", sim_dark_games_clip.item())
print("CLIP: Dark Souls III vs Team Fortress 2 (dark vs bright, cartoonish):", sim_dark_vs_bright_clip.item())

CLIP: Dark Souls III vs Fallout 4 (both dark, gritty): 0.611577033996582
CLIP: Dark Souls III vs Team Fortress 2 (dark vs bright, cartoonish): 0.6021046042442322


In [29]:
from transformers import CLIPTokenizerFast

clip_tokenizer = CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")

genre_prompts = [
    "a dark and gritty video game",
    "a colorful and cartoonish video game",
]

text_inputs = clip_tokenizer(genre_prompts, padding=True, return_tensors="pt").to(device)

with torch.no_grad():
    text_outputs = clip_model.text_model(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"]
    )
    text_pooled = text_outputs.pooler_output
    text_embeds = clip_model.text_projection(text_pooled)

text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

def genre_scores(app_id):
    img_vec = get_clip_image_vec(app_id).to(device)
    img_vec_norm = img_vec / img_vec.norm(dim=-1, keepdim=True)
    scores = (img_vec_norm @ text_embeds.T).squeeze(0)
    return scores.cpu().numpy()

for name, app_id in [("Dark Souls III", darksouls_app_id), ("Fallout 4", fallout4_app_id), ("Team Fortress 2", tf2_app_id)]:
    scores = genre_scores(app_id)
    print(f"{name}: dark/gritty={scores[0]:.4f}, colorful/cartoonish={scores[1]:.4f}")

Dark Souls III: dark/gritty=0.2676, colorful/cartoonish=0.2249
Fallout 4: dark/gritty=0.2496, colorful/cartoonish=0.2444
Team Fortress 2: dark/gritty=0.2781, colorful/cartoonish=0.2554


In [30]:
import random

def has_tag(tags, target_tag):
    return isinstance(tags, list) and target_tag in tags

horror_app_ids = metadata_filtered[metadata_filtered["tags"].apply(lambda t: has_tag(t, "Horror"))]["app_id"].tolist()
family_app_ids = metadata_filtered[metadata_filtered["tags"].apply(lambda t: has_tag(t, "Family Friendly"))]["app_id"].tolist()

print("Games tagged Horror:", len(horror_app_ids))
print("Games tagged Family Friendly:", len(family_app_ids))

random.seed(42)
sample_horror = random.sample(horror_app_ids, min(150, len(horror_app_ids)))
sample_family = random.sample(family_app_ids, min(150, len(family_app_ids)))

def average_genre_scores(app_id_list):
    all_scores = []
    for app_id in app_id_list:
        if app_id in game_id_map:
            scores = genre_scores(app_id)
            all_scores.append(scores)
    all_scores = np.array(all_scores)
    return all_scores.mean(axis=0)

horror_avg = average_genre_scores(sample_horror)
family_avg = average_genre_scores(sample_family)

print()
print(f"Horror games average: dark/gritty={horror_avg[0]:.4f}, colorful/cartoonish={horror_avg[1]:.4f}")
print(f"Family Friendly games average: dark/gritty={family_avg[0]:.4f}, colorful/cartoonish={family_avg[1]:.4f}")

Games tagged Horror: 4104
Games tagged Family Friendly: 3493

Horror games average: dark/gritty=0.2523, colorful/cartoonish=0.2295
Family Friendly games average: dark/gritty=0.2442, colorful/cartoonish=0.2521


In [31]:
def win_rate(app_id_list, winning_index):
    wins = 0
    total = 0
    for app_id in app_id_list:
        if app_id in game_id_map:
            scores = genre_scores(app_id)
            if scores[winning_index] > scores[1 - winning_index]:
                wins += 1
            total += 1
    return wins / total, total

horror_win_rate, horror_n = win_rate(sample_horror, winning_index=0)
family_win_rate, family_n = win_rate(sample_family, winning_index=1)

print(f"Horror games where dark/gritty beat colorful/cartoonish: {horror_win_rate:.1%} ({horror_n} games checked)")
print(f"Family Friendly games where colorful/cartoonish beat dark/gritty: {family_win_rate:.1%} ({family_n} games checked)")

Horror games where dark/gritty beat colorful/cartoonish: 84.7% (150 games checked)
Family Friendly games where colorful/cartoonish beat dark/gritty: 64.0% (150 games checked)


In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F

game_text_embeddings = torch.load("/kaggle/working/game_text_embeddings.pt").to(device)
game_clip_embeddings_tensor = torch.load("/kaggle/working/game_clip_embeddings.pt").to(device)

gnn_encoder.eval()
with torch.no_grad():
    x_dict = {
        'user': user_embedding.weight,
        'game': game_embedding.weight,
    }
    out_dict = gnn_encoder(x_dict, data.edge_index_dict)
    gnn_game_embeddings = out_dict['game']

print("GNN embeddings shape:", gnn_game_embeddings.shape)
print("NLP embeddings shape:", game_text_embeddings.shape)
print("CLIP embeddings shape:", game_clip_embeddings_tensor.shape)

gnn_norm = F.normalize(gnn_game_embeddings, p=2, dim=1)
nlp_norm = F.normalize(game_text_embeddings, p=2, dim=1)
clip_norm = F.normalize(game_clip_embeddings_tensor, p=2, dim=1)

fused_input = torch.cat([gnn_norm, nlp_norm, clip_norm], dim=1)
print("Concatenated fused input shape:", fused_input.shape)

class FusionLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

fusion_layer = FusionLayer(input_dim=960, hidden_dim=256, output_dim=128).to(device)

fused_game_embeddings = fusion_layer(fused_input)
print("Fused game embeddings shape:", fused_game_embeddings.shape)

GNN embeddings shape: torch.Size([36229, 64])
NLP embeddings shape: torch.Size([36229, 384])
CLIP embeddings shape: torch.Size([36229, 512])
Concatenated fused input shape: torch.Size([36229, 960])
Fused game embeddings shape: torch.Size([36229, 128])


In [33]:
torch.save(gnn_encoder.state_dict(), "/kaggle/working/gnn_encoder.pt")
torch.save(user_embedding.state_dict(), "/kaggle/working/user_embedding.pt")
torch.save(game_embedding.state_dict(), "/kaggle/working/game_embedding.pt")
torch.save(fusion_layer.state_dict(), "/kaggle/working/fusion_layer.pt")
print("All model weights saved")

All model weights saved


In [34]:
class UserProjection(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)

user_projection = UserProjection(input_dim=64, output_dim=128).to(device)

# fused_input is already fixed, GNN + NLP + CLIP concatenated, no need to recompute it
fused_input = fused_input.detach()
gnn_user_embeddings = out_dict['user'].detach()

fusion_params = list(fusion_layer.parameters()) + list(user_projection.parameters())
optimizer = optim.Adam(fusion_params, lr=0.01)

edge_label_index = train_data['user', 'reviews', 'game'].edge_label_index
edge_label = train_data['user', 'reviews', 'game'].edge_label

val_edge_label_index = val_data['user', 'reviews', 'game'].edge_label_index
val_edge_label = val_data['user', 'reviews', 'game'].edge_label

EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    fusion_layer.train()
    user_projection.train()
    optimizer.zero_grad()

    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)

    user_idx = edge_label_index[0]
    game_idx = edge_label_index[1]
    scores = (user_out[user_idx] * fused_game_out[game_idx]).sum(dim=-1)

    loss = F.binary_cross_entropy_with_logits(scores, edge_label)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}")

fusion_layer.eval()
user_projection.eval()
with torch.no_grad():
    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)

    val_user_idx = val_edge_label_index[0]
    val_game_idx = val_edge_label_index[1]
    val_scores = (user_out[val_user_idx] * fused_game_out[val_game_idx]).sum(dim=-1).cpu().numpy()
    val_labels = val_edge_label.cpu().numpy()

    fusion_auc = roc_auc_score(val_labels, val_scores)
    print(f"Fusion Model Validation AUC: {fusion_auc:.4f}")

torch.save(fusion_layer.state_dict(), "/kaggle/working/fusion_layer.pt")
torch.save(user_projection.state_dict(), "/kaggle/working/user_projection.pt")
print("Fusion layer and user projection saved")

Epoch 1, Train Loss: 0.6926
Epoch 10, Train Loss: 0.4258
Epoch 20, Train Loss: 0.3572
Epoch 30, Train Loss: 0.3212
Epoch 40, Train Loss: 0.3148
Epoch 50, Train Loss: 0.3128
Epoch 60, Train Loss: 0.3102
Epoch 70, Train Loss: 0.3090
Epoch 80, Train Loss: 0.3081
Epoch 90, Train Loss: 0.3074
Epoch 100, Train Loss: 0.3068
Fusion Model Validation AUC: 0.9390
Fusion layer and user projection saved


In [35]:
sample_user_ids = [0, 500, 10000]

with torch.no_grad():
    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)

    recommendations_by_user_fusion = {}
    for uid in sample_user_ids:
        user_vec = user_out[uid]
        scores = (fused_game_out * user_vec).sum(dim=-1)
        top_k = torch.topk(scores, k=10)
        top_game_node_ids = top_k.indices.cpu().numpy()
        top_app_ids = [reverse_game_map[gid] for gid in top_game_node_ids]
        titles = games[games["app_id"].isin(top_app_ids)]["title"].tolist()
        recommendations_by_user_fusion[uid] = set(top_app_ids)
        print(f"User {uid} top 10 (fusion):", titles)
        print()

overlap_0_500 = recommendations_by_user_fusion[0] & recommendations_by_user_fusion[500]
overlap_0_10000 = recommendations_by_user_fusion[0] & recommendations_by_user_fusion[10000]
overlap_500_10000 = recommendations_by_user_fusion[500] & recommendations_by_user_fusion[10000]

print("Fusion overlap, user 0 vs 500:", len(overlap_0_500), "out of 10")
print("Fusion overlap, user 0 vs 10000:", len(overlap_0_10000), "out of 10")
print("Fusion overlap, user 500 vs 10000:", len(overlap_500_10000), "out of 10")
print()
print("For comparison, GNN-only overlap was: 6/10, 5/10, 2/10")

User 0 top 10 (fusion): ['Half-Life 2', 'Left 4 Dead 2', 'Red Dead Redemption 2', 'The Witcher® 3: Wild Hunt', 'Metro Exodus', 'Portal 2', 'DARK SOULS™ III', 'Team Fortress 2', 'Kingdom Come: Deliverance', 'DOOM']

User 500 top 10 (fusion): ['Half-Life 2', 'Left 4 Dead 2', 'The Witcher® 3: Wild Hunt', 'Metro Exodus', 'Portal 2', 'DARK SOULS™ III', 'Team Fortress 2', 'Kingdom Come: Deliverance', 'DOOM', 'Resident Evil 2']

User 10000 top 10 (fusion): ['The Witcher® 3: Wild Hunt', 'Metro Exodus', "No Man's Sky", 'DARK SOULS™ III', 'Team Fortress 2', 'BioShock Infinite', 'Far Cry 3', 'Kingdom Come: Deliverance', 'DOOM', 'Devil May Cry 5']

Fusion overlap, user 0 vs 500: 9 out of 10
Fusion overlap, user 0 vs 10000: 6 out of 10
Fusion overlap, user 500 vs 10000: 6 out of 10

For comparison, GNN-only overlap was: 6/10, 5/10, 2/10


In [36]:
with torch.no_grad():
    fused_std = fused_game_out.std(dim=0).mean().item()
    print("Average per-dimension std across all fused game vectors:", fused_std)

    def get_fused_vec(title):
        app_id = games[games["title"] == title].iloc[0]["app_id"]
        node_id = game_id_map[app_id]
        return fused_game_out[node_id].unsqueeze(0)

    v1 = get_fused_vec("Fallout 4")
    v2 = get_fused_vec("DARK SOULS™ III")
    v3 = get_fused_vec("Team Fortress 2")

    print("Fallout 4 vs Dark Souls III fused similarity:", cosine_similarity(v1, v2).item())
    print("Fallout 4 vs Team Fortress 2 fused similarity:", cosine_similarity(v1, v3).item())
    print("Dark Souls III vs Team Fortress 2 fused similarity:", cosine_similarity(v2, v3).item())

    user_std = user_out.std(dim=0).mean().item()
    print("Average per-dimension std across all user vectors:", user_std)

Average per-dimension std across all fused game vectors: 0.09391605108976364
Fallout 4 vs Dark Souls III fused similarity: 0.9941215515136719
Fallout 4 vs Team Fortress 2 fused similarity: 0.9959027171134949
Dark Souls III vs Team Fortress 2 fused similarity: 0.9989190101623535
Average per-dimension std across all user vectors: 0.15290699899196625


In [37]:
from torch_geometric.utils import negative_sampling

# Reinitialize fresh, the collapsed weights aren't worth continuing from
fusion_layer = FusionLayer(input_dim=960, hidden_dim=256, output_dim=128).to(device)
user_projection = UserProjection(input_dim=64, output_dim=128).to(device)

fusion_params = list(fusion_layer.parameters()) + list(user_projection.parameters())
optimizer = optim.Adam(fusion_params, lr=0.001, weight_decay=1e-5)

TEMPERATURE = 0.1
train_edge_index = train_data['user', 'reviews', 'game'].edge_index

EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    fusion_layer.train()
    user_projection.train()
    optimizer.zero_grad()

    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)

    # Normalize to unit length, removes the magnitude shortcut entirely
    fused_game_norm = F.normalize(fused_game_out, dim=-1)
    user_norm = F.normalize(user_out, dim=-1)

    # Fresh negative samples every single epoch, no reusing the same fixed fakes
    neg_edge_index = negative_sampling(
        edge_index=train_edge_index,
        num_nodes=(n_users, n_games),
        num_neg_samples=train_edge_index.size(1)
    )

    pos_user, pos_game = train_edge_index[0], train_edge_index[1]
    neg_user, neg_game = neg_edge_index[0], neg_edge_index[1]

    pos_score = (user_norm[pos_user] * fused_game_norm[pos_game]).sum(dim=-1) / TEMPERATURE
    neg_score = (user_norm[neg_user] * fused_game_norm[neg_game]).sum(dim=-1) / TEMPERATURE

    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)])

    loss = F.binary_cross_entropy_with_logits(scores, labels)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}")

# Diagnostic check, same as before, run immediately
fusion_layer.eval()
user_projection.eval()
with torch.no_grad():
    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)
    fused_game_norm = F.normalize(fused_game_out, dim=-1)

    fused_std = fused_game_norm.std(dim=0).mean().item()
    print("\nAverage per-dimension std across fused game vectors:", fused_std)

    v1 = get_fused_vec("Fallout 4")
    v2 = get_fused_vec("DARK SOULS™ III")
    v3 = get_fused_vec("Team Fortress 2")
    print("Fallout 4 vs Dark Souls III:", cosine_similarity(v1, v2).item())
    print("Fallout 4 vs Team Fortress 2:", cosine_similarity(v1, v3).item())
    print("Dark Souls III vs Team Fortress 2:", cosine_similarity(v2, v3).item())

Epoch 1, Train Loss: 0.7243
Epoch 10, Train Loss: 0.4443
Epoch 20, Train Loss: 0.3370
Epoch 30, Train Loss: 0.3311
Epoch 40, Train Loss: 0.3228
Epoch 50, Train Loss: 0.3187
Epoch 60, Train Loss: 0.3164
Epoch 70, Train Loss: 0.3137
Epoch 80, Train Loss: 0.3119
Epoch 90, Train Loss: 0.3107
Epoch 100, Train Loss: 0.3098

Average per-dimension std across fused game vectors: 0.02901417762041092
Fallout 4 vs Dark Souls III: 0.9769014120101929
Fallout 4 vs Team Fortress 2: 0.9830018281936646
Dark Souls III vs Team Fortress 2: 0.978438138961792


In [38]:
BATCH_SIZE = 1024
EPOCHS = 5

fusion_layer = FusionLayer(input_dim=960, hidden_dim=256, output_dim=128).to(device)
user_projection = UserProjection(input_dim=64, output_dim=128).to(device)

fusion_params = list(fusion_layer.parameters()) + list(user_projection.parameters())
optimizer = optim.Adam(fusion_params, lr=0.001, weight_decay=1e-5)

TEMPERATURE = 0.1

train_edge_index = train_data['user', 'reviews', 'game'].edge_index
pos_user_all = train_edge_index[0]
pos_game_all = train_edge_index[1]
n_train_edges = pos_user_all.size(0)

for epoch in range(1, EPOCHS + 1):
    fusion_layer.train()
    user_projection.train()

    perm = torch.randperm(n_train_edges, device=device)
    pos_user_shuffled = pos_user_all[perm]
    pos_game_shuffled = pos_game_all[perm]

    total_loss = 0
    n_batches = 0

    for start in range(0, n_train_edges, BATCH_SIZE):
        batch_user_idx = pos_user_shuffled[start:start + BATCH_SIZE]
        batch_game_idx = pos_game_shuffled[start:start + BATCH_SIZE]

        if batch_user_idx.size(0) < 2:
            continue

        optimizer.zero_grad()

        batch_game_out = fusion_layer(fused_input[batch_game_idx])
        batch_user_out = user_projection(gnn_user_embeddings[batch_user_idx])

        batch_game_norm = F.normalize(batch_game_out, dim=-1)
        batch_user_norm = F.normalize(batch_user_out, dim=-1)

        logits = (batch_user_norm @ batch_game_norm.T) / TEMPERATURE
        labels = torch.arange(batch_user_norm.size(0), device=device)

        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    print(f"Epoch {epoch}, Avg Loss: {total_loss / n_batches:.4f}")

# Full diagnostic check afterward, same as before
fusion_layer.eval()
user_projection.eval()
with torch.no_grad():
    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)
    fused_game_norm = F.normalize(fused_game_out, dim=-1)

    fused_std = fused_game_norm.std(dim=0).mean().item()
    print("\nAverage per-dimension std across fused game vectors:", fused_std)

    v1 = get_fused_vec("Fallout 4")
    v2 = get_fused_vec("DARK SOULS™ III")
    v3 = get_fused_vec("Team Fortress 2")
    print("Fallout 4 vs Dark Souls III:", cosine_similarity(v1, v2).item())
    print("Fallout 4 vs Team Fortress 2:", cosine_similarity(v1, v3).item())
    print("Dark Souls III vs Team Fortress 2:", cosine_similarity(v2, v3).item())

Epoch 1, Avg Loss: 6.5150
Epoch 2, Avg Loss: 6.4201
Epoch 3, Avg Loss: 6.3877
Epoch 4, Avg Loss: 6.3681
Epoch 5, Avg Loss: 6.3557

Average per-dimension std across fused game vectors: 0.07623356580734253
Fallout 4 vs Dark Souls III: 0.8113083839416504
Fallout 4 vs Team Fortress 2: 0.7397061586380005
Dark Souls III vs Team Fortress 2: 0.7064687013626099


In [39]:
EPOCHS = 25

for epoch in range(1, EPOCHS + 1):
    fusion_layer.train()
    user_projection.train()

    perm = torch.randperm(n_train_edges, device=device)
    pos_user_shuffled = pos_user_all[perm]
    pos_game_shuffled = pos_game_all[perm]

    total_loss = 0
    n_batches = 0

    for start in range(0, n_train_edges, BATCH_SIZE):
        batch_user_idx = pos_user_shuffled[start:start + BATCH_SIZE]
        batch_game_idx = pos_game_shuffled[start:start + BATCH_SIZE]

        if batch_user_idx.size(0) < 2:
            continue

        optimizer.zero_grad()

        batch_game_out = fusion_layer(fused_input[batch_game_idx])
        batch_user_out = user_projection(gnn_user_embeddings[batch_user_idx])

        batch_game_norm = F.normalize(batch_game_out, dim=-1)
        batch_user_norm = F.normalize(batch_user_out, dim=-1)

        logits = (batch_user_norm @ batch_game_norm.T) / TEMPERATURE
        labels = torch.arange(batch_user_norm.size(0), device=device)

        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    print(f"Epoch {epoch}, Avg Loss: {total_loss / n_batches:.4f}")

# Same diagnostic check again
fusion_layer.eval()
user_projection.eval()
with torch.no_grad():
    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)
    fused_game_norm = F.normalize(fused_game_out, dim=-1)

    fused_std = fused_game_norm.std(dim=0).mean().item()
    print("\nAverage per-dimension std across fused game vectors:", fused_std)

    v1 = get_fused_vec("Fallout 4")
    v2 = get_fused_vec("DARK SOULS™ III")
    v3 = get_fused_vec("Team Fortress 2")
    print("Fallout 4 vs Dark Souls III:", cosine_similarity(v1, v2).item())
    print("Fallout 4 vs Team Fortress 2:", cosine_similarity(v1, v3).item())
    print("Dark Souls III vs Team Fortress 2:", cosine_similarity(v2, v3).item())

Epoch 1, Avg Loss: 6.3466
Epoch 2, Avg Loss: 6.3393
Epoch 3, Avg Loss: 6.3336
Epoch 4, Avg Loss: 6.3290
Epoch 5, Avg Loss: 6.3244
Epoch 6, Avg Loss: 6.3215
Epoch 7, Avg Loss: 6.3180
Epoch 8, Avg Loss: 6.3154
Epoch 9, Avg Loss: 6.3129
Epoch 10, Avg Loss: 6.3107
Epoch 11, Avg Loss: 6.3089
Epoch 12, Avg Loss: 6.3069
Epoch 13, Avg Loss: 6.3050
Epoch 14, Avg Loss: 6.3037
Epoch 15, Avg Loss: 6.3024
Epoch 16, Avg Loss: 6.3009
Epoch 17, Avg Loss: 6.2997
Epoch 18, Avg Loss: 6.2986
Epoch 19, Avg Loss: 6.2974
Epoch 20, Avg Loss: 6.2962
Epoch 21, Avg Loss: 6.2953
Epoch 22, Avg Loss: 6.2946
Epoch 23, Avg Loss: 6.2936
Epoch 24, Avg Loss: 6.2924
Epoch 25, Avg Loss: 6.2919

Average per-dimension std across fused game vectors: 0.07511796057224274
Fallout 4 vs Dark Souls III: 0.8487223386764526
Fallout 4 vs Team Fortress 2: 0.7661487460136414
Dark Souls III vs Team Fortress 2: 0.7836599946022034


In [40]:
import random

def get_fused_vec_by_app_id(app_id):
    node_id = game_id_map[app_id]
    return fused_game_norm[node_id].unsqueeze(0)

random.seed(42)
sample_horror = [a for a in horror_app_ids if a in game_id_map]
sample_family = [a for a in family_app_ids if a in game_id_map]
sample_horror = random.sample(sample_horror, min(200, len(sample_horror)))
sample_family = random.sample(sample_family, min(200, len(sample_family)))

def avg_pairwise_similarity(list_a, list_b, n_pairs=300):
    sims = []
    for _ in range(n_pairs):
        a = random.choice(list_a)
        b = random.choice(list_b)
        va = get_fused_vec_by_app_id(a)
        vb = get_fused_vec_by_app_id(b)
        sims.append(cosine_similarity(va, vb).item())
    return sum(sims) / len(sims)

within_horror = avg_pairwise_similarity(sample_horror, sample_horror)
within_family = avg_pairwise_similarity(sample_family, sample_family)
cross_genre = avg_pairwise_similarity(sample_horror, sample_family)

print("Average similarity, Horror vs Horror (same genre):", within_horror)
print("Average similarity, Family Friendly vs Family Friendly (same genre):", within_family)
print("Average similarity, Horror vs Family Friendly (different genre):", cross_genre)

Average similarity, Horror vs Horror (same genre): 0.2510807795325915
Average similarity, Family Friendly vs Family Friendly (same genre): 0.17166597903395692
Average similarity, Horror vs Family Friendly (different genre): 0.18386630694692332


In [41]:
sample_user_ids = [0, 500, 10000]

with torch.no_grad():
    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)
    fused_game_norm = F.normalize(fused_game_out, dim=-1)
    user_norm = F.normalize(user_out, dim=-1)

    recommendations_by_user_fusion2 = {}
    for uid in sample_user_ids:
        user_vec = user_norm[uid]
        scores = (fused_game_norm * user_vec).sum(dim=-1)
        top_k = torch.topk(scores, k=10)
        top_game_node_ids = top_k.indices.cpu().numpy()
        top_app_ids = [reverse_game_map[gid] for gid in top_game_node_ids]
        titles = games[games["app_id"].isin(top_app_ids)]["title"].tolist()
        recommendations_by_user_fusion2[uid] = set(top_app_ids)
        print(f"User {uid} top 10 (fusion, fixed):", titles)
        print()

overlap_0_500 = recommendations_by_user_fusion2[0] & recommendations_by_user_fusion2[500]
overlap_0_10000 = recommendations_by_user_fusion2[0] & recommendations_by_user_fusion2[10000]
overlap_500_10000 = recommendations_by_user_fusion2[500] & recommendations_by_user_fusion2[10000]

print("Fixed fusion overlap, user 0 vs 500:", len(overlap_0_500), "out of 10")
print("Fixed fusion overlap, user 0 vs 10000:", len(overlap_0_10000), "out of 10")
print("Fixed fusion overlap, user 500 vs 10000:", len(overlap_500_10000), "out of 10")
print()
print("Broken fusion (before fix) was: 10/10, 9/10, 9/10")
print("GNN-only (Phase 2) was: 6/10, 5/10, 2/10")

User 0 top 10 (fusion, fixed): ['Maelstrom', 'King of Crabs', 'Blazing Sails', 'Eternal Return', 'Survivalist: Invisible Strain', 'Longvinter', '瘟疫清零计划 Project Zero', 'Chair Simulator', 'POLYGON', 'Soulstone Survivors']

User 500 top 10 (fusion, fixed): ['Spermination', 'Call of Duty®', "Garry's Mod", 'Call of Duty®: Black Ops III', 'Hogs of War', 'Sven Co-op', 'Call of Duty: World at War', 'Call of Duty®: Black Ops', "KovaaK's", 'Madden NFL 23']

User 10000 top 10 (fusion, fixed): ['War Of The Zombie', 'The Underground Man 2', 'Antisnake', 'Professional Construction - The Simulation', 'Incorp Inc', 'Reborn Tribe', 'The Flesh God', 'Storm over the Pacific', 'Prison Tycoon 3™: Lockdown', 'TrainFormation']

Fixed fusion overlap, user 0 vs 500: 0 out of 10
Fixed fusion overlap, user 0 vs 10000: 0 out of 10
Fixed fusion overlap, user 500 vs 10000: 0 out of 10

Broken fusion (before fix) was: 10/10, 9/10, 9/10
GNN-only (Phase 2) was: 6/10, 5/10, 2/10


In [42]:
val_edge_label_index = val_data['user', 'reviews', 'game'].edge_label_index
val_edge_label = val_data['user', 'reviews', 'game'].edge_label

pos_mask = val_edge_label == 1
val_pos_users = val_edge_label_index[0][pos_mask]
val_pos_games = val_edge_label_index[1][pos_mask]

print("Total held-out positive validation pairs:", val_pos_users.size(0))

SAMPLE_SIZE = 2000
perm = torch.randperm(val_pos_users.size(0))[:SAMPLE_SIZE]
eval_users = val_pos_users[perm]
eval_true_games = val_pos_games[perm]

K = 10

with torch.no_grad():
    fused_game_out = fusion_layer(fused_input)
    user_out = user_projection(gnn_user_embeddings)
    fused_game_norm = F.normalize(fused_game_out, dim=-1)
    user_norm = F.normalize(user_out, dim=-1)

    hits = 0
    for i in range(SAMPLE_SIZE):
        u = eval_users[i]
        true_game = eval_true_games[i].item()

        scores = (fused_game_norm * user_norm[u]).sum(dim=-1)
        top_k = torch.topk(scores, k=K).indices.cpu().numpy()

        if true_game in top_k:
            hits += 1

recall_at_k = hits / SAMPLE_SIZE
print(f"Recall@{K} on {SAMPLE_SIZE} held-out validation pairs: {recall_at_k:.4f}")

Total held-out positive validation pairs: 343976
Recall@10 on 2000 held-out validation pairs: 0.0105


In [43]:
def evaluate_recall_at_k(sample_size=2000, k=10, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    perm = torch.randperm(val_pos_users.size(0))[:sample_size]
    eval_users = val_pos_users[perm]
    eval_true_games = val_pos_games[perm]

    fusion_layer.eval()
    user_projection.eval()
    with torch.no_grad():
        fused_game_out = fusion_layer(fused_input)
        user_out = user_projection(gnn_user_embeddings)
        fused_game_norm = F.normalize(fused_game_out, dim=-1)
        user_norm = F.normalize(user_out, dim=-1)

        hits = 0
        for i in range(sample_size):
            u = eval_users[i]
            true_game = eval_true_games[i].item()
            scores = (fused_game_norm * user_norm[u]).sum(dim=-1)
            top_k = torch.topk(scores, k=k).indices.cpu().numpy()
            if true_game in top_k:
                hits += 1
    return hits / sample_size

best_recall = 0.0
patience = 3
epochs_without_improvement = 0
CHECK_EVERY = 20
MAX_EPOCHS = 300

epoch = 0
while epoch < MAX_EPOCHS:
    for _ in range(CHECK_EVERY):
        epoch += 1
        fusion_layer.train()
        user_projection.train()

        perm = torch.randperm(n_train_edges, device=device)
        pos_user_shuffled = pos_user_all[perm]
        pos_game_shuffled = pos_game_all[perm]

        total_loss = 0
        n_batches = 0
        for start in range(0, n_train_edges, BATCH_SIZE):
            batch_user_idx = pos_user_shuffled[start:start + BATCH_SIZE]
            batch_game_idx = pos_game_shuffled[start:start + BATCH_SIZE]
            if batch_user_idx.size(0) < 2:
                continue

            optimizer.zero_grad()
            batch_game_out = fusion_layer(fused_input[batch_game_idx])
            batch_user_out = user_projection(gnn_user_embeddings[batch_user_idx])
            batch_game_norm = F.normalize(batch_game_out, dim=-1)
            batch_user_norm = F.normalize(batch_user_out, dim=-1)

            logits = (batch_user_norm @ batch_game_norm.T) / TEMPERATURE
            labels = torch.arange(batch_user_norm.size(0), device=device)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

    recall = evaluate_recall_at_k(sample_size=2000, k=10, seed=42)
    print(f"Epoch {epoch}, Avg Loss: {total_loss / n_batches:.4f}, Recall@10: {recall:.4f}")

    if recall > best_recall:
        best_recall = recall
        epochs_without_improvement = 0
        torch.save(fusion_layer.state_dict(), "/kaggle/working/fusion_layer_best.pt")
        torch.save(user_projection.state_dict(), "/kaggle/working/user_projection_best.pt")
        print("  New best, saved checkpoint")
    else:
        epochs_without_improvement += 1
        print(f"  No improvement, patience {epochs_without_improvement}/{patience}")
        if epochs_without_improvement >= patience:
            print("Stopping early, recall has plateaued")
            break

print(f"\nFinal best Recall@10: {best_recall:.4f}")

Epoch 20, Avg Loss: 6.2811, Recall@10: 0.0160
  New best, saved checkpoint
Epoch 40, Avg Loss: 6.2755, Recall@10: 0.0185
  New best, saved checkpoint
Epoch 60, Avg Loss: 6.2717, Recall@10: 0.0160
  No improvement, patience 1/3
Epoch 80, Avg Loss: 6.2689, Recall@10: 0.0150
  No improvement, patience 2/3
Epoch 100, Avg Loss: 6.2667, Recall@10: 0.0145
  No improvement, patience 3/3
Stopping early, recall has plateaued

Final best Recall@10: 0.0185


In [44]:
fusion_layer_final = FusionLayer(input_dim=960, hidden_dim=256, output_dim=128).to(device)
user_projection_final = UserProjection(input_dim=64, output_dim=128).to(device)

fusion_layer_final.load_state_dict(torch.load("/kaggle/working/fusion_layer_best.pt"))
user_projection_final.load_state_dict(torch.load("/kaggle/working/user_projection_best.pt"))
fusion_layer_final.eval()
user_projection_final.eval()

with torch.no_grad():
    fused_game_out_final = fusion_layer_final(fused_input)
    user_out_final = user_projection_final(gnn_user_embeddings)
    fused_game_norm_final = F.normalize(fused_game_out_final, dim=-1)
    user_norm_final = F.normalize(user_out_final, dim=-1)

LARGE_SAMPLE = 20000
torch.manual_seed(123)
perm = torch.randperm(val_pos_users.size(0))[:LARGE_SAMPLE]
eval_users_large = val_pos_users[perm]
eval_true_games_large = val_pos_games[perm]

hits = 0
with torch.no_grad():
    for i in range(LARGE_SAMPLE):
        u = eval_users_large[i]
        true_game = eval_true_games_large[i].item()
        scores = (fused_game_norm_final * user_norm_final[u]).sum(dim=-1)
        top_k = torch.topk(scores, k=10).indices.cpu().numpy()
        if true_game in top_k:
            hits += 1

final_recall_at_10 = hits / LARGE_SAMPLE
print(f"Final Recall@10 on {LARGE_SAMPLE} held-out validation pairs: {final_recall_at_10:.4f}")

random_baseline = 10 / n_games
print(f"Random baseline Recall@10 would be approximately: {random_baseline:.5f}")
print(f"Model performs approximately {final_recall_at_10 / random_baseline:.1f}x better than random")

Final Recall@10 on 20000 held-out validation pairs: 0.0112
Random baseline Recall@10 would be approximately: 0.00028
Model performs approximately 40.4x better than random


In [45]:
torch.save(fusion_layer_final.state_dict(), "/kaggle/working/fusion_layer_final.pt")
torch.save(user_projection_final.state_dict(), "/kaggle/working/user_projection_final.pt")
print("Final Phase 3 models saved")

Final Phase 3 models saved
